In [5]:
import matplotlib.pyplot as plt
import numpy as np

In [13]:
def plot_window(sigma, delta, tau, center=None, span=None, xlim=None, ylim=None, figsize=(9, 9), simplified=False):
    """
    Plot the strips defined by sigma, delta, and tau in the (s, t)-plane.

    Parameters
    ----------
    sigma : tuple[float, float]
        Interval for s, defining the vertical strip V.
    delta : tuple[float, float]
        Interval for t - s, defining the diagonal strip D.
    tau : tuple[float, float]
        Interval for t, defining the horizontal strip H.
    center : tuple[float, float], optional
        Center of the viewing window. If omitted, the center is inferred
        from sigma and tau.
    span : float or tuple[float, float], optional
        Width and height of the viewing window. A scalar gives equal spans
        in both directions.
    xlim, ylim : tuple[float, float], optional
        Explicit plot limits. If both are given, they override center/span.
    figsize : tuple[float, float]
        Matplotlib figure size.
    simplified : bool
        If True, suppress axis labels and the legend and enlarge tick labels.
    """
    s_min, s_max = sigma
    d_min, d_max = delta
    t_min, t_max = tau

    fig, ax = plt.subplots(figsize=figsize)

    # Determine the plotting window. Explicit limits take precedence over the center/span representation.
    if xlim is not None and ylim is not None:
        plot_min_x, plot_max_x = xlim
        plot_min_y, plot_max_y = ylim
    else:
        if center is None:
            center_x = (s_min + s_max) / 2
            center_y = (t_min + t_max) / 2
        else:
            center_x, center_y = center

        if span is None:
            width = s_max - s_min
            height = t_max - t_min
            span_x = span_y = 2.5 * max(width, height)
        elif isinstance(span, (int, float)):
            span_x = span_y = span
        else:
            span_x, span_y = span

        plot_min_x = center_x - span_x / 2
        plot_max_x = center_x + span_x / 2
        plot_min_y = center_y - span_y / 2
        plot_max_y = center_y + span_y / 2

    view_width = plot_max_x - plot_min_x
    view_height = plot_max_y - plot_min_y

    # x-coordinates used to draw the diagonal strip D: d_min <= t - s <= d_max.
    x_range = np.linspace(plot_min_x, plot_max_x, 200)

    # Plot the diagonal strip D.
    ax.fill_between(
        x_range,
        x_range + d_min,
        x_range + d_max,
        color="gray",
        alpha=0.15,
        label=r"$D$",
    )

    # Draw the two boundaries of D.
    ax.plot(
        x_range,
        x_range + d_min,
        "k--",
        alpha=0.3,
        linewidth=1,
    )
    ax.plot(
        x_range,
        x_range + d_max,
        "k--",
        alpha=0.3,
        linewidth=1,
    )

    # Draw the interval sigma along the bottom of the viewing window.
    ax.hlines(
        plot_min_y + 0.05 * view_height,
        s_min,
        s_max,
        color="red",
        linewidth=5,
        label=r"$V$",
    )

    # Draw the interval tau along the left side of the viewing window.
    ax.vlines(
        plot_min_x + 0.05 * view_width,
        t_min,
        t_max,
        color="green",
        linewidth=5,
        label=r"$H$",
    )

    # The intersection of V and H is represented by the blue rectangle.
    rect = plt.Rectangle(
        (s_min, t_min),
        s_max - s_min,
        t_max - t_min,
        linewidth=2,
        edgecolor="blue",
        facecolor="blue",
        alpha=0.1,
        label=r"$V \cap H$",
    )
    ax.add_patch(rect)

    # Compute and shade the portion of V \cap H that also lies inside D.
    x_fill = np.linspace(s_min, s_max, 500)
    y_lower = np.maximum(t_min, x_fill + d_min)
    y_upper = np.minimum(t_max, x_fill + d_max)

    ax.fill_between(
        x_fill,
        y_lower,
        y_upper,
        where=(y_upper >= y_lower),
        color="darkviolet",
        alpha=0.5,
        label=r"$V \cap H \cap D$",
    )

    # Plot formatting.
    ax.set_aspect("equal")
    ax.set_xlim(plot_min_x, plot_max_x)
    ax.set_ylim(plot_min_y, plot_max_y)
    ax.grid(True, linestyle=":", alpha=0.6)

    if simplified:
        ax.tick_params(axis="both", which="major", labelsize=20)
    else:
        ax.set_xlabel("s", fontsize=12)
        ax.set_ylabel("t", fontsize=12)
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

In [12]:
def plot_linking(R, center=None, span=None, xlim=None, ylim=None, figsize=(9, 9), simplified=False):
    """
    Plot a finite temporal linking R as points in the (s, t)-plane.

    Parameters
    ----------
    R : iterable[tuple[float, float]]
        Points (s, t) belonging to the relation.
    center : tuple[float, float], optional
        Center of the viewing window. If omitted, it is inferred from R.
    span : float or tuple[float, float], optional
        Width and height of the viewing window. A scalar gives equal spans
        in both directions.
    xlim, ylim : tuple[float, float], optional
        Explicit plot limits. If both are given, they override center/span.
    figsize : tuple[float, float]
        Matplotlib figure size.
    simplified : bool
        If True, use larger axis labels and tick labels and omit the legend.
    """
    if len(R) == 0:
        raise ValueError("R must contain at least one point.")

    # Sort by the s-coordinate so that an optional connecting path proceeds
    # from left to right.
    R_sorted = sorted(R, key=lambda point: point[0])

    s_vals = np.array([s for s, _ in R_sorted])
    t_vals = np.array([t for _, t in R_sorted])

    fig, ax = plt.subplots(figsize=figsize)

    # Determine the plotting window. Explicit limits take precedence over
    # the center/span representation.
    if xlim is not None and ylim is not None:
        plot_min_x, plot_max_x = xlim
        plot_min_y, plot_max_y = ylim
    else:
        if center is None:
            center_x = (s_vals.min() + s_vals.max()) / 2
            center_y = (t_vals.min() + t_vals.max()) / 2
        else:
            center_x, center_y = center

        if span is None:
            # Avoid a zero-size window when all points have the same
            # s- or t-coordinate.
            width = max(1e-9, s_vals.max() - s_vals.min())
            height = max(1e-9, t_vals.max() - t_vals.min())
            span_x = span_y = 1.5 * max(width, height)
        elif isinstance(span, (int, float)):
            span_x = span_y = span
        else:
            span_x, span_y = span

        plot_min_x = center_x - span_x / 2
        plot_max_x = center_x + span_x / 2
        plot_min_y = center_y - span_y / 2
        plot_max_y = center_y + span_y / 2

    # Plot the points belonging to R.
    ax.scatter(
        s_vals,
        t_vals,
        color="darkviolet",
        s=100,
        zorder=2,
        label=r"$R$",
    )

    # Plot formatting.
    ax.set_aspect("equal")
    ax.set_xlim(plot_min_x, plot_max_x)
    ax.set_ylim(plot_min_y, plot_max_y)
    ax.grid(True, linestyle=":", alpha=0.6)

    if simplified:
        ax.tick_params(axis="both", which="major", labelsize=20)
        ax.set_xlabel("s", fontsize=25)
        ax.set_ylabel("t", fontsize=25)
    else:
        ax.set_xlabel("s", fontsize=18)
        ax.set_ylabel("t", fontsize=18)
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()